# 05 — Curriculum Training

Pulls curriculum JSONL files from HF Hub, trains a fresh GPT-2 model in curriculum phases,
pushes per-epoch checkpoints to Hub (enables resume), and plots training curves.

**Resume:** If the pod is interrupted, re-run all cells. Cell 5 will find the latest
`phase_XX_epoch_YY` checkpoint on Hub and pick up from there.

In [ ]:
# ── Cell 1: RunPod setup (run once per session) ──
# Uncomment and run this block when starting a fresh RunPod session.

# !git clone https://github.com/flakoash/influece_driven_curriculum_sorter.git /workspace/babylm
# %cd /workspace/babylm
# !pip install -e ".[dev]" --quiet

print("Setup complete.")

In [ ]:
# ── Cell 2: Config — edit these before running ──

TOKENIZER_NAME      = "BabyLM-community/BabyLM-2026-Baseline-GPT2-Strict-Small"
HUB_CURRICULUM_REPO = None   # "yourname/babylm-curriculum-run001" — source of epoch_XX.jsonl files
HUB_MODEL_REPO      = None   # "yourname/babylm-trained-run001"    — destination for checkpoints
HF_TOKEN            = None   # or: import os; HF_TOKEN = os.environ["HF_TOKEN"]

PHASE_EPOCHS         = [5, 5]   # epochs per curriculum phase (one entry per epoch_XX.jsonl)
MAX_SEQ_LEN          = 128
SEED                 = 0
PASS_BATCH_SIZE      = 64    # per-device batch size (A40 48 GB safe)
EFFECTIVE_BATCH_SIZE = 256
LEARNING_RATE        = 7e-4

print(f"Phases        : {PHASE_EPOCHS}  ({sum(PHASE_EPOCHS)} total epochs)")
print(f"Curriculum Hub: {HUB_CURRICULUM_REPO}")
print(f"Model Hub     : {HUB_MODEL_REPO}")

In [ ]:
# ── Cell 3: Imports + device ──
import os, sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

for _candidate in [Path("../src"), Path("src"), Path("/workspace/babylm/src")]:
    if _candidate.exists() and str(_candidate.resolve()) not in sys.path:
        sys.path.insert(0, str(_candidate.resolve()))

from influence_curriculum.train import TrainingConfig, train_curriculum

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT    = Path("/workspace/outputs/curriculum_training")
OUT.mkdir(parents=True, exist_ok=True)
CURRICULUM_DIR = Path("/workspace/curriculum")
CURRICULUM_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
# ── Cell 4: Pull curriculum JSONL files from Hub ──
# Downloads epoch_00.jsonl, epoch_01.jsonl, ... into /workspace/curriculum/
# Skips files already present.

phase_files = []

if HUB_CURRICULUM_REPO:
    from huggingface_hub import hf_hub_download, list_repo_files
    remote_files = [f for f in list_repo_files(HUB_CURRICULUM_REPO, repo_type="dataset", token=HF_TOKEN)
                    if f.startswith("curriculum/epoch_") and f.endswith(".jsonl")]
    remote_files = sorted(remote_files)
    print(f"Found {len(remote_files)} curriculum files on Hub: {remote_files}")
    for remote_path in remote_files:
        fname = Path(remote_path).name
        local = CURRICULUM_DIR / fname
        if not local.exists():
            hf_hub_download(HUB_CURRICULUM_REPO, remote_path, local_dir=str(CURRICULUM_DIR),
                            repo_type="dataset", token=HF_TOKEN)
            print(f"  downloaded {fname}")
        else:
            print(f"  {fname} already present, skipping")
        phase_files.append(str(local))
else:
    # Fallback: use local files if Hub not set
    phase_files = sorted(str(p) for p in CURRICULUM_DIR.glob("epoch_*.jsonl"))
    print(f"HUB_CURRICULUM_REPO not set — using local files: {[Path(p).name for p in phase_files]}")

assert len(phase_files) >= len(PHASE_EPOCHS), (
    f"Need {len(PHASE_EPOCHS)} JSONL files for PHASE_EPOCHS={PHASE_EPOCHS}, "
    f"found {len(phase_files)}: {phase_files}"
)
phase_files = phase_files[:len(PHASE_EPOCHS)]
phases = list(zip(phase_files, PHASE_EPOCHS))
print(f"\nTraining phases:")
for i, (f, e) in enumerate(phases):
    n = Path(f).read_text().count('\n')
    print(f"  Phase {i}: {Path(f).name}  {n:,} docs  {e} epochs")

In [ ]:
# ── Cell 5: Resume check ──
# Scans HUB_MODEL_REPO for the latest phase_XX_epoch_YY checkpoint.
# Sets start_phase / start_epoch and loads weights if a checkpoint is found.

import re

start_phase, start_epoch = 0, 0
model = GPT2LMHeadModel(GPT2Config(
    n_embd=384, n_layer=8, n_head=6, n_inner=1536,
    vocab_size=16384, n_positions=1024,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
))
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

if HUB_MODEL_REPO:
    try:
        from huggingface_hub import list_repo_files, snapshot_download
        ckpt_pattern = re.compile(r"^phase_(\d+)_epoch_(\d+)/config\.json$")
        remote_ckpts = [
            (int(m.group(1)), int(m.group(2)))
            for f in list_repo_files(HUB_MODEL_REPO, repo_type="model", token=HF_TOKEN)
            if (m := ckpt_pattern.match(f))
        ]
        if remote_ckpts:
            latest_phase, latest_epoch = max(remote_ckpts)
            ckpt_name = f"phase_{latest_phase:02d}_epoch_{latest_epoch:02d}"
            local_ckpt = OUT / "checkpoints" / ckpt_name
            if not local_ckpt.exists():
                print(f"Downloading checkpoint {ckpt_name} from Hub...")
                snapshot_download(
                    repo_id=HUB_MODEL_REPO,
                    local_dir=str(OUT / "checkpoints"),
                    allow_patterns=f"{ckpt_name}/*",
                    local_dir_use_symlinks=False,
                    repo_type="model",
                    token=HF_TOKEN,
                )
            model = GPT2LMHeadModel.from_pretrained(str(local_ckpt))
            # advance resume point past the completed epoch
            n_epochs_in_phase = PHASE_EPOCHS[latest_phase]
            if latest_epoch + 1 < n_epochs_in_phase:
                start_phase, start_epoch = latest_phase, latest_epoch + 1
            else:
                start_phase, start_epoch = latest_phase + 1, 0
            print(f"Resuming from {ckpt_name} → start_phase={start_phase}, start_epoch={start_epoch}")
        else:
            print("No checkpoints found on Hub — starting from scratch.")
    except Exception as e:
        print(f"Hub resume check failed ({e}) — starting from scratch.")
else:
    print("HUB_MODEL_REPO not set — starting from scratch, no Hub push.")

In [ ]:
# ── Cell 6: Train ──

train_cfg = TrainingConfig(
    per_device_batch_size=PASS_BATCH_SIZE,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    max_seq_len=MAX_SEQ_LEN,
    fp16=(DEVICE == "cuda"),
)

checkpoint_paths, history = train_curriculum(
    model, tokenizer,
    phases=phases,
    output_dir=str(OUT),
    config=train_cfg,
    seed=SEED,
    device=DEVICE,
    start_phase=start_phase,
    start_epoch=start_epoch,
    hub_repo=HUB_MODEL_REPO,
    hub_token=HF_TOKEN,
)
print(f"\nTraining complete. {len(checkpoint_paths)} checkpoints saved.")

In [ ]:
# ── Cell 7: Push final model to Hub ──

if HUB_MODEL_REPO and checkpoint_paths:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id=HUB_MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
    final_ckpt = checkpoint_paths[-1]
    print(f"Pushing final model ({Path(final_ckpt).name}) to {HUB_MODEL_REPO} ...")
    api.upload_folder(
        folder_path=final_ckpt,
        repo_id=HUB_MODEL_REPO,
        path_in_repo="final",
        repo_type="model",
        token=HF_TOKEN,
        commit_message="final trained model",
    )
    print("Done.")
else:
    print("HUB_MODEL_REPO not set or no checkpoints — skipping Hub push.")

In [ ]:
# ── Cell 8: Training curves ──
import math
import matplotlib.pyplot as plt

epochs_x    = list(range(len(history)))
losses      = [h["loss"] for h in history]
perplexities = [math.exp(min(l, 20)) for l in losses]  # cap at exp(20) to avoid overflow
lrs         = [h["lr"] for h in history]
boundary    = PHASE_EPOCHS[0] - 0.5   # vertical line between phase 0 and phase 1

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(epochs_x, losses, marker="o", color="steelblue")
axes[0].axvline(boundary, color="red", linestyle="--", alpha=0.6, label="phase boundary")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(epochs_x, perplexities, marker="o", color="darkorange")
axes[1].axvline(boundary, color="red", linestyle="--", alpha=0.6)
axes[1].set_ylabel("Perplexity")
axes[1].set_title("Perplexity (exp(loss))")

axes[2].plot(epochs_x, lrs, marker="o", color="green")
axes[2].axvline(boundary, color="red", linestyle="--", alpha=0.6)
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("LR Schedule")
axes[2].set_xlabel("Epoch (global)")

plt.tight_layout()
plt.savefig(str(OUT / "training_curves.png"), dpi=150)
plt.show()
print(f"Curves saved to {OUT / 'training_curves.png'}")